In [5]:
import sympy as sp

First, we only look at attitude estimation. In order to predict the state propagation, we model the system by non-linear state-space functions. We use the following **state**, **input** and **output vector**:

\begin{align}
    \mathbf{x} &= \begin{bmatrix}q_w & \mathbf{q}_v\end{bmatrix}^T = \begin{bmatrix}q_w & q_x & q_y & q_z\end{bmatrix}^T \\
    \mathbf{u} &= \begin{bmatrix}\omega_x & \omega_y & \omega_z\end{bmatrix}^T \\
    \mathbf{y} &= \begin{bmatrix}a_x & a_y & a_z & m_x & m_y & m_z\end{bmatrix}^T
\end{align}

We also use the following **transition** and  **output function**:

\begin{align}
    \mathbf{f}(\mathbf{x}_{t-1}, \mathbf{u}_{t}) &= \Big(\mathbf{I}_4 + \frac{\Delta t}{2}\boldsymbol\Omega_t\Big)\mathbf{x}_{t-1} \\
    \mathbf{h}(\mathbf{x}_{t}, \mathbf{u}_{t}) &= \begin{bmatrix}\hat{\mathbf{a}} \\ \hat{\mathbf{m}}\end{bmatrix}
= \begin{bmatrix}\mathbf{C}(\hat{\mathbf{x}_{t}})^T \mathbf{g} \\ \mathbf{C}(\hat{\mathbf{x}_{t}})^T \mathbf{r}\end{bmatrix}
\end{align}

where

\begin{align}
    \boldsymbol\Omega_t &=
        \begin{bmatrix}
            0 & -\boldsymbol\omega^T\\
            \boldsymbol\omega & -\lfloor\boldsymbol\omega\rfloor_\times
        \end{bmatrix} =
        \begin{bmatrix}
            0 & -\omega_x & -\omega_y & -\omega_z \\
            \omega_x & 0 & \omega_z & -\omega_y \\
            \omega_y & -\omega_z & 0 & \omega_x \\
            \omega_z & \omega_y & -\omega_x & 0
        \end{bmatrix} \\

    \mathbf{C}(\hat{\mathbf{x}}) &=
        \begin{bmatrix}
            \hat{q}_w^2 + \hat{q}_x^2 - \hat{q}_y^2 - \hat{q}_z^2 & 2(\hat{q}_x\hat{q}_y-\hat{q}_w\hat{q}_z) & 2(\hat{q}_x\hat{q}_z+\hat{q}_w\hat{q}_y) \\
            2(\hat{q}_x\hat{q}_y+\hat{q}_w\hat{q}_z) & \hat{q}_w^2 - \hat{q}_x^2 + \hat{q}_y^2 - \hat{q}_z^2 & 2(\hat{q}_y\hat{q}_z-\hat{q}_w\hat{q}_x) \\
            2(\hat{q}_x\hat{q}_z-\hat{q}_w\hat{q}_y) & 2(\hat{q}_w\hat{q}_x+\hat{q}_y\hat{q}_z) & \hat{q}_w^2 - \hat{q}_x^2 - \hat{q}_y^2 + \hat{q}_z^2
        \end{bmatrix}
\end{align}

In [27]:
# create state and input vector
qw, qx, qy, qz = sp.symbols('q_w, q_x, q_y, q_z')
wx, wy, wz = sp.symbols('\\omega_x, \\omega_y, \\omega_z')

x = sp.Matrix([qw, qx, qy, qz])
u = sp.Matrix([wx, wy, wz])

# create omega matrix
dt = sp.Symbol('\\Delta t')
Omega = lambda u: sp.Matrix([
    [0, -u[0], -u[1], -u[2]],
    [u[0], 0, u[2], -u[1]],
    [u[1], -u[2], 0, u[0]],
    [u[2], u[1], -u[0], 0]
])

In [25]:
# create state transition function
f = lambda x, u: (sp.eye(4) + dt/2*Omega(u)) @ x
f(x, u)

Matrix([
[-\Delta t*\omega_x*q_x/2 - \Delta t*\omega_y*q_y/2 - \Delta t*\omega_z*q_z/2 + q_w],
[ \Delta t*\omega_x*q_w/2 - \Delta t*\omega_y*q_z/2 + \Delta t*\omega_z*q_y/2 + q_x],
[ \Delta t*\omega_x*q_z/2 + \Delta t*\omega_y*q_w/2 - \Delta t*\omega_z*q_x/2 + q_y],
[-\Delta t*\omega_x*q_y/2 + \Delta t*\omega_y*q_x/2 + \Delta t*\omega_z*q_w/2 + q_z]])

In [26]:
# compute jacobian
F = lambda x, u: f(x, u).jacobian(x)
F(x, u)

Matrix([
[                  1, -\Delta t*\omega_x/2, -\Delta t*\omega_y/2, -\Delta t*\omega_z/2],
[\Delta t*\omega_x/2,                    1,  \Delta t*\omega_z/2, -\Delta t*\omega_y/2],
[\Delta t*\omega_y/2, -\Delta t*\omega_z/2,                    1,  \Delta t*\omega_x/2],
[\Delta t*\omega_z/2,  \Delta t*\omega_y/2, -\Delta t*\omega_x/2,                    1]])

In [39]:
# create quaternion rotation function
C = lambda q: sp.Quaternion(*q, norm=1).to_rotation_matrix()

# create constants
g = sp.Matrix([0, 0, 1])

# create output function
h = lambda x, u: C(x).T @ g
h(x, u)

Matrix([
[           -2*q_w*q_y + 2*q_x*q_z],
[            2*q_w*q_x + 2*q_y*q_z],
[q_w**2 - q_x**2 - q_y**2 + q_z**2]])

In [40]:
# compute jacobian
H = lambda x, u: h(x, u).jacobian(x)
H(x, u)

Matrix([
[-2*q_y,  2*q_z, -2*q_w, 2*q_x],
[ 2*q_x,  2*q_w,  2*q_z, 2*q_y],
[ 2*q_w, -2*q_x, -2*q_y, 2*q_z]])